In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools

import kagglehub

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)

# Téléchargez le jeu de données depuis Kaggle
path = kagglehub.dataset_download("fahadullaha/facial-emotion-recognition-dataset")
print("Path to dataset files:", path)

data_dir = os.path.join(path, "processed_data")
print("Using data_dir:", data_dir)


In [ ]:
# ============================================================
# 5. Build MobileNetV2 model (transfer learning)
# ============================================================
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)


# for i, layer in enumerate(base_model.layers):
#     print(i, layer.name)

base_model.trainable = False  # freeze base model first

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = layers.Dropout(0.3, name="dropout")(x)
x = layers.Dense(256, activation="relu", name="fc1")(x)
x = layers.Dropout(0.5, name="dropout1")(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

model = keras.Model(inputs, outputs, name="MobileNetV2_emotion")
model.summary()

initial_epochs = 50
base_learning_rate = 1e-3

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=base_learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_frozen = model.fit(
    train_ds,
    epochs=initial_epochs,
    validation_data=val_ds
)

base_model.trainable = True

fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.summary()

fine_tune_epochs = 50
total_epochs = initial_epochs + fine_tune_epochs

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


early_stop_ft = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)


history_finetune = model.fit(
    train_ds,
    epochs=total_epochs,
    initial_epoch=history_frozen.epoch[-1],
    validation_data=val_ds,
    callbacks=[early_stop_ft],
)
